# 03 - Reward Modeling

This notebook covers reward model training:
- Reward model architecture
- Bradley-Terry preference learning
- Training and evaluation
- Reward distribution analysis

In [ ]:
import sys
sys.path.append('..')

import torch
import matplotlib.pyplot as plt
from datasets import load_dataset

from reward.reward_model import RewardModel, RewardModelConfig
from reward.reward_trainer import RewardTrainer, RewardTrainingConfig
from reward.reward_analysis import RewardAnalyzer

## 1. Understanding Reward Models

A reward model learns to predict human preferences. It takes a prompt-response pair and outputs a scalar reward score.

In [ ]:
# Reward model configuration
config = RewardModelConfig(
    base_model_name="EleutherAI/pythia-410m",
    hidden_dim=None,  # Use model's hidden size
    num_layers=1,
    dropout=0.1,
    load_in_4bit=False,  # Set True for QLoRA
)

print(f"Reward model config: {config}")

## 2. Load Preference Dataset

In [ ]:
# Load preference data
dataset = load_dataset('Anthropic/hh-rlhf', split='train[:2000]')
eval_dataset = load_dataset('Anthropic/hh-rlhf', split='test[:500]')

print(f"Training samples: {len(dataset)}")
print(f"Evaluation samples: {len(eval_dataset)}")

# Show sample
sample = dataset[0]
print(f"\nChosen (first 200 chars): {sample['chosen'][:200]}")
print(f"\nRejected (first 200 chars): {sample['rejected'][:200]}")

## 3. Bradley-Terry Loss

The reward model is trained using Bradley-Terry preference modeling:

$$P(\text{chosen} > \text{rejected}) = \sigma(r(\text{chosen}) - r(\text{rejected}))$$

Loss: $-\log \sigma(r_\text{chosen} - r_\text{rejected})$

In [ ]:
def bradley_terry_loss(chosen_rewards, rejected_rewards):
    """Compute Bradley-Terry preference loss."""
    return -torch.log(torch.sigmoid(chosen_rewards - rejected_rewards)).mean()

# Example
chosen = torch.tensor([2.0, 1.5, 3.0])
rejected = torch.tensor([1.0, 1.0, 2.0])

loss = bradley_terry_loss(chosen, rejected)
print(f"Example loss: {loss.item():.4f}")
print(f"Win probability: {torch.sigmoid(chosen - rejected).mean().item():.4f}")

## 4. Training Configuration

In [ ]:
training_config = RewardTrainingConfig(
    output_dir="../outputs/reward_demo",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=1e-5,
    loss_type="bradley_terry",
    margin=0.0,
)

print(f"Training config: {training_config}")

## 5. Training Loop

In [ ]:
# Initialize trainer (implementation required)
# trainer = RewardTrainer(
#     model=reward_model,
#     config=training_config,
#     train_dataset=dataset,
#     eval_dataset=eval_dataset,
# )

# Train
# metrics = trainer.train()

print("Training would execute here...")
print("Expected metrics:")
print("- loss: Bradley-Terry loss value")
print("- accuracy: Percentage where chosen > rejected")
print("- reward_margin: Average gap between chosen and rejected")

## 6. Reward Distribution Analysis

In [ ]:
# Analyze reward distribution (after training)
# analyzer = RewardAnalyzer(reward_model=reward_model)
# distribution = analyzer.analyze_distribution(eval_dataset)

# Simulated distribution for visualization
import numpy as np

chosen_rewards = np.random.normal(1.5, 0.5, 500)
rejected_rewards = np.random.normal(0.5, 0.5, 500)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Distribution plot
axes[0].hist(chosen_rewards, bins=30, alpha=0.7, label='Chosen', color='green')
axes[0].hist(rejected_rewards, bins=30, alpha=0.7, label='Rejected', color='red')
axes[0].set_xlabel('Reward Score')
axes[0].set_ylabel('Count')
axes[0].legend()
axes[0].set_title('Reward Distribution')

# Margin distribution
margins = chosen_rewards - rejected_rewards
axes[1].hist(margins, bins=30, alpha=0.7, color='blue')
axes[1].axvline(x=0, color='red', linestyle='--', label='Tie')
axes[1].set_xlabel('Reward Margin (Chosen - Rejected)')
axes[1].set_ylabel('Count')
axes[1].legend()
axes[1].set_title(f'Margin Distribution (Accuracy: {(margins > 0).mean()*100:.1f}%)')

plt.tight_layout()
plt.show()

## 7. Next Steps

- Use the reward model for PPO training in `04_ppo_vs_dpo.ipynb`
- Analyze reward hacking in `05_reward_hacking_analysis.ipynb`
- Monitor for length correlation (potential hacking indicator)